# Geographic Variation in Real Estate Marketing Language

## Part 1: Core Analysis - Vocabulary Overlap & Categorization

This notebook performs the core analysis for the research paper:
- Loads discriminative words for each city
- Calculates vocabulary overlap (Jaccard similarity)
- Categorizes words thematically
- Creates visualizations

**Expected Runtime:** 2-3 minutes

## Setup: Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully")

## 1. Load Discriminative Words

These words were extracted using log-odds ratio with Dirichlet prior (Monroe et al. 2008).
Each city has separate word lists for fast-selling vs slow-selling properties.

In [ ]:
# Configuration
WORD_COUNTS_DIR = "dataset/word_counts"
OUTPUT_DIR = "result/geographic_analysis"
PERCENTAGE = 0.25  # 25th/75th percentile threshold
N_WORDS = 50  # Top 50 discriminative words per group

# City and property type mappings
CITIES = ["CH", "NY", "LA"]
CITY_NAMES = {"CH": "Chicago", "NY": "New York", "LA": "Los Angeles"}
PROPERTY_TYPES = {0: "Single Family", 1: "Condo/Townhouse"}

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Configuration:")
print(f"  - Cities: {', '.join([CITY_NAMES[c] for c in CITIES])}")
print(f"  - Threshold: {int(PERCENTAGE*100)}th percentile")
print(f"  - Words per group: {N_WORDS}")
print(f"  - Output directory: {OUTPUT_DIR}")

In [ ]:
# Load discriminative words for all cities and property types
fast_words = {}  # {(city, type, pct): [(word, zscore), ...]}
slow_words = {}  # {(city, type, pct): [(word, zscore), ...]}

for city in CITIES:
    for prop_type in [0, 1]:
        # Fast-selling words (group_0)
        fast_file = os.path.join(
            WORD_COUNTS_DIR,
            str(PERCENTAGE),
            f"{city}_{prop_type}_group_0_zscore.csv"
        )
        
        # Slow-selling words (group_2)
        slow_file = os.path.join(
            WORD_COUNTS_DIR,
            str(PERCENTAGE),
            f"{city}_{prop_type}_group_2_zscore.csv"
        )
        
        try:
            fast_df = pd.read_csv(fast_file, header=None, names=["word", "zscore"])
            fast_words[(city, prop_type, PERCENTAGE)] = list(
                zip(fast_df['word'].head(N_WORDS), fast_df['zscore'].head(N_WORDS))
            )
            
            slow_df = pd.read_csv(slow_file, header=None, names=["word", "zscore"])
            slow_words[(city, prop_type, PERCENTAGE)] = list(
                zip(slow_df['word'].head(N_WORDS), slow_df['zscore'].head(N_WORDS))
            )
            
            print(f"✅ Loaded {city} {PROPERTY_TYPES[prop_type]}")
        except FileNotFoundError as e:
            print(f"⚠️  Warning: Could not load file: {e}")

print(f"\n✅ Loaded discriminative words for {len(fast_words)} city/type combinations")

### Preview: Top Words by City

In [ ]:
# Display top 10 fast-selling words for each city (single-family homes)
print("Top 10 Fast-Selling Words (Single-Family Homes):\n")

for city in CITIES:
    key = (city, 0, PERCENTAGE)  # Single-family = 0
    if key in fast_words:
        print(f"\n{CITY_NAMES[city]}:")
        print("-" * 50)
        words_list = fast_words[key][:10]
        for i, (word, zscore) in enumerate(words_list, 1):
            print(f"{i:2d}. {word:20s} (z-score: {zscore:.3f})")

## 2. Calculate Vocabulary Overlap (Jaccard Similarity)

Jaccard similarity measures how many words are shared between two cities:
$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

- J = 0: No shared words (completely distinct)
- J = 1: Identical vocabularies
- J < 0.1: Minimal similarity (our finding!)

In [ ]:
def calculate_vocabulary_overlap():
    """Calculate Jaccard similarity between cities' discriminative vocabularies."""
    results = []
    
    for prop_type in [0, 1]:
        for sale_type in ['fast', 'slow']:
            words_dict = fast_words if sale_type == 'fast' else slow_words
            
            # Get word sets for each city
            city_words = {}
            for city in CITIES:
                key = (city, prop_type, PERCENTAGE)
                if key in words_dict:
                    city_words[city] = set([w for w, _ in words_dict[key]])
            
            # Calculate pairwise Jaccard similarity
            for i, city1 in enumerate(CITIES):
                for city2 in CITIES[i+1:]:
                    if city1 in city_words and city2 in city_words:
                        intersection = len(city_words[city1] & city_words[city2])
                        union = len(city_words[city1] | city_words[city2])
                        jaccard = intersection / union if union > 0 else 0
                        
                        results.append({
                            'Property Type': PROPERTY_TYPES[prop_type],
                            'Sale Speed': sale_type.capitalize(),
                            'City 1': CITY_NAMES[city1],
                            'City 2': CITY_NAMES[city2],
                            'Jaccard Similarity': jaccard,
                            'Shared Words': intersection,
                            'Total Unique Words': union,
                            'Overlap %': (intersection / min(
                                len(city_words[city1]),
                                len(city_words[city2])
                            ) * 100) if min(len(city_words[city1]), len(city_words[city2])) > 0 else 0
                        })
    
    return pd.DataFrame(results)

# Calculate overlap
overlap_df = calculate_vocabulary_overlap()

# Save results
overlap_df.to_csv(f"{OUTPUT_DIR}/vocabulary_overlap.csv", index=False)
print(f"✅ Saved vocabulary overlap to {OUTPUT_DIR}/vocabulary_overlap.csv\n")

# Display results
print("VOCABULARY OVERLAP RESULTS")
print("=" * 80)
overlap_df

In [ ]:
# Summary statistics
print("\nSummary: Jaccard Similarity by Property Type and Sale Speed")
print("=" * 80)
summary = overlap_df.groupby(['Property Type', 'Sale Speed'])['Jaccard Similarity'].agg([
    ('Mean', 'mean'),
    ('Std', 'std'),
    ('Min', 'min'),
    ('Max', 'max')
]).round(3)
print(summary)

print("\n🔑 KEY FINDING: Cities share only 3-9% of vocabulary (extremely low overlap!)")

## 3. Identify Unique City Words

Words that appear in one city's top 50 but NOT in the other cities' top 50.

In [ ]:
def get_unique_city_words(top_n=20):
    """Identify words unique to each city."""
    unique_words = {}
    
    for prop_type in [0, 1]:
        for sale_type in ['fast', 'slow']:
            words_dict = fast_words if sale_type == 'fast' else slow_words
            
            # Get word sets for each city
            city_word_sets = {}
            city_word_scores = {}
            
            for city in CITIES:
                key = (city, prop_type, PERCENTAGE)
                if key in words_dict:
                    city_word_sets[city] = set([w for w, _ in words_dict[key]])
                    city_word_scores[city] = {w: s for w, s in words_dict[key]}
            
            # Find unique words for each city
            for city in CITIES:
                if city not in city_word_sets:
                    continue
                
                other_cities = [c for c in CITIES if c != city]
                other_words = set()
                for other_city in other_cities:
                    if other_city in city_word_sets:
                        other_words |= city_word_sets[other_city]
                
                # Words in this city but not in others
                unique = city_word_sets[city] - other_words
                
                # Sort by z-score
                unique_scored = [
                    (w, city_word_scores[city][w]) for w in unique
                    if w in city_word_scores[city]
                ]
                unique_scored.sort(key=lambda x: x[1], reverse=True)
                
                key_label = f"{CITY_NAMES[city]}_{PROPERTY_TYPES[prop_type]}_{sale_type}"
                unique_words[key_label] = unique_scored[:top_n]
    
    return unique_words

# Get unique words
unique_words = get_unique_city_words(top_n=20)

# Save to JSON
unique_serializable = {
    k: [(w, float(s)) for w, s in v] for k, v in unique_words.items()
}
with open(f"{OUTPUT_DIR}/unique_city_words.json", 'w') as f:
    json.dump(unique_serializable, f, indent=2)
    
print(f"✅ Saved unique city words to {OUTPUT_DIR}/unique_city_words.json")

In [ ]:
# Display unique words for fast-selling single-family homes
print("Top 15 UNIQUE Words (Fast-Selling Single-Family Homes)")
print("=" * 80)

for city in CITIES:
    key = f"{CITY_NAMES[city]}_Single Family_fast"
    if key in unique_words and unique_words[key]:
        print(f"\n{CITY_NAMES[city].upper()}:")
        print("-" * 50)
        for i, (word, score) in enumerate(unique_words[key][:15], 1):
            print(f"{i:2d}. {word:25s} z-score: {score:6.3f}")

## 4. Thematic Categorization

Categorize discriminative words into themes:
- **Location/Neighborhood**: Geographic references
- **Transit/Accessibility**: Public transportation
- **Property Features**: Rooms, spaces
- **Condition/Quality**: Renovation, maintenance
- **Amenities**: Interior features, systems
- **Outdoor/Views**: Exterior spaces, vistas
- **Investment/Market**: Financial terms
- **School/Family**: Education, family features
- **Specific Locations**: Neighborhood names

In [ ]:
# Define category keywords
categories = {
    'Location/Neighborhood': [
        'downtown', 'neighborhood', 'district', 'area', 'block', 'street',
        'ave', 'avenue', 'road', 'drive', 'place', 'way', 'north', 'south',
        'east', 'west', 'central', 'uptown', 'midtown'
    ],
    'Transit/Accessibility': [
        'metra', 'cta', 'subway', 'train', 'bus', 'transit', 'transportation',
        'walk', 'walkable', 'walkability', 'bike', 'commute', 'accessible',
        'station', 'stop', 'line', 'lirr', 'metro'
    ],
    'Property Features': [
        'bedroom', 'bathroom', 'kitchen', 'living', 'dining', 'room', 'space',
        'floor', 'ceiling', 'window', 'door', 'closet', 'storage', 'garage',
        'parking', 'basement', 'attic', 'patio', 'deck', 'balcony', 'yard',
        'garden', 'backyard', 'frontyard'
    ],
    'Condition/Quality': [
        'new', 'renovated', 'updated', 'upgraded', 'remodeled', 'modern',
        'contemporary', 'luxury', 'pristine', 'immaculate', 'mint', 'turnkey',
        'move-in', 'ready', 'finished', 'polished', 'maintained', 'restored',
        'refurbished', 'rehab', 'fixer', 'as-is', 'tlc', 'potential', 'rehabbers'
    ],
    'Amenities': [
        'pool', 'spa', 'gym', 'fitness', 'doorman', 'concierge', 'elevator',
        'laundry', 'washer', 'dryer', 'dishwasher', 'ac', 'heating', 'hvac',
        'central-air', 'fireplace', 'hardwood', 'carpet', 'tile', 'granite',
        'marble', 'stainless', 'appliances', 'wifi', 'smart-home', 'doorperson'
    ],
    'Outdoor/Views': [
        'view', 'views', 'skyline', 'waterfront', 'lakefront', 'river',
        'ocean', 'beach', 'mountain', 'park', 'green', 'trees', 'nature',
        'outdoor', 'patio', 'terrace', 'rooftop', 'deck', 'sunroom',
        'hardscaping', 'landscap'
    ],
    'Investment/Market': [
        'investment', 'opportunity', 'potential', 'investor', 'rental',
        'income', 'cash-flow', 'roi', 'appreciation', 'equity', 'bidding',
        'offer', 'price', 'value', 'deal', 'motivated', 'must-see',
        "won't-last", 'hot', 'priced-to-sell', 'adu'
    ],
    'School/Family': [
        'school', 'schools', 'district', 'rated', 'family', 'kid',
        'children', 'playground', 'park', 'safe', 'quiet', 'residential',
        'neighborhood', 'community', 'csun'
    ],
}

print("Category definitions created")
print(f"Total categories: {len(categories)}")

In [ ]:
def categorize_words():
    """Categorize discriminative words thematically."""
    results = []
    
    for prop_type in [0, 1]:
        for sale_type in ['fast', 'slow']:
            words_dict = fast_words if sale_type == 'fast' else slow_words
            
            for city in CITIES:
                key = (city, prop_type, PERCENTAGE)
                if key not in words_dict:
                    continue
                
                words_with_scores = words_dict[key]
                
                for word, zscore in words_with_scores:
                    # Find category
                    word_lower = word.lower().replace('_', '-')
                    category = 'Specific Locations'  # Default
                    
                    for cat_name, keywords in categories.items():
                        if any(kw in word_lower for kw in keywords):
                            category = cat_name
                            break
                    
                    results.append({
                        'City': CITY_NAMES[city],
                        'Property Type': PROPERTY_TYPES[prop_type],
                        'Sale Speed': sale_type.capitalize(),
                        'Word': word,
                        'Z-Score': zscore,
                        'Category': category
                    })
    
    return pd.DataFrame(results)

# Categorize words
categorized_df = categorize_words()

# Save results
categorized_df.to_csv(f"{OUTPUT_DIR}/word_categories.csv", index=False)
print(f"✅ Saved word categorization to {OUTPUT_DIR}/word_categories.csv")

print(f"\nTotal words categorized: {len(categorized_df)}")
print(f"\nCategory distribution (all words):")
print(categorized_df['Category'].value_counts())

In [ ]:
# Analyze category distribution by city
distribution = categorized_df.groupby([
    'City', 'Property Type', 'Sale Speed', 'Category'
]).size().reset_index(name='Word Count')

# Calculate percentages
total_by_group = categorized_df.groupby([
    'City', 'Property Type', 'Sale Speed'
]).size().reset_index(name='Total')

distribution = distribution.merge(
    total_by_group,
    on=['City', 'Property Type', 'Sale Speed']
)
distribution['Percentage'] = (distribution['Word Count'] / distribution['Total'] * 100).round(2)

# Save
distribution.to_csv(f"{OUTPUT_DIR}/category_distribution.csv", index=False)
print(f"✅ Saved category distribution to {OUTPUT_DIR}/category_distribution.csv")

# Display for fast-selling single-family
print("\nCategory Distribution: Fast-Selling Single-Family Homes")
print("=" * 80)
subset = distribution[
    (distribution['Property Type'] == 'Single Family') &
    (distribution['Sale Speed'] == 'Fast')
][['City', 'Category', 'Word Count', 'Percentage']].sort_values(['City', 'Percentage'], ascending=[True, False])
subset

## 5. Visualizations

In [ ]:
# Figure 1: Vocabulary Overlap Heatmap
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Vocabulary Overlap Across Cities (Jaccard Similarity)', 
             fontsize=16, fontweight='bold')

for idx, (prop_type, sale_type) in enumerate([
    ('Single Family', 'Fast'),
    ('Single Family', 'Slow'),
    ('Condo/Townhouse', 'Fast'),
    ('Condo/Townhouse', 'Slow')
]):
    ax = axes[idx // 2, idx % 2]
    
    subset = overlap_df[
        (overlap_df['Property Type'] == prop_type) &
        (overlap_df['Sale Speed'] == sale_type)
    ]
    
    if not subset.empty:
        # Create matrix for heatmap
        cities = ['Chicago', 'New York', 'Los Angeles']
        matrix = np.zeros((3, 3))
        
        for _, row in subset.iterrows():
            i = cities.index(row['City 1'])
            j = cities.index(row['City 2'])
            matrix[i, j] = row['Jaccard Similarity']
            matrix[j, i] = row['Jaccard Similarity']
        
        # Set diagonal to 1 (self-similarity)
        np.fill_diagonal(matrix, 1.0)
        
        sns.heatmap(
            matrix,
            annot=True,
            fmt='.3f',
            xticklabels=cities,
            yticklabels=cities,
            cmap='RdYlGn',
            vmin=0,
            vmax=1,
            ax=ax,
            cbar_kws={'label': 'Jaccard Similarity'}
        )
        ax.set_title(f'{prop_type} - {sale_type}-Selling', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/vocabulary_overlap_heatmap.png", dpi=300, bbox_inches='tight')
print(f"✅ Saved: {OUTPUT_DIR}/vocabulary_overlap_heatmap.png")
plt.show()

In [ ]:
# Figure 2: Category Distribution by City
subset = distribution[
    (distribution['Property Type'] == 'Single Family') &
    (distribution['Sale Speed'] == 'Fast')
]

if not subset.empty:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    pivot_data = subset.pivot(
        index='Category',
        columns='City',
        values='Percentage'
    ).fillna(0)
    
    pivot_data.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('Category Distribution: Fast-Selling Single Family Homes by City',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Word Category', fontsize=12)
    ax.set_ylabel('Percentage of Discriminative Words', fontsize=12)
    ax.legend(title='City', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/category_distribution_by_city.png", dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {OUTPUT_DIR}/category_distribution_by_city.png")
    plt.show()

In [ ]:
# Figure 3: Top Unique Words per City
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.suptitle('Top City-Specific Words (Fast-Selling Single Family)', 
             fontsize=16, fontweight='bold')

for idx, city_code in enumerate(['CH', 'NY', 'LA']):
    city_name = CITY_NAMES[city_code]
    key = f"{city_name}_Single Family_fast"
    
    if key in unique_words and unique_words[key]:
        words, scores = zip(*unique_words[key][:15])
        
        axes[idx].barh(range(len(words)), scores, color=f'C{idx}')
        axes[idx].set_yticks(range(len(words)))
        axes[idx].set_yticklabels(words)
        axes[idx].invert_yaxis()
        axes[idx].set_xlabel('Z-Score', fontsize=11)
        axes[idx].set_title(city_name, fontsize=13, fontweight='bold')
        axes[idx].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/unique_words_by_city.png", dpi=300, bbox_inches='tight')
print(f"✅ Saved: {OUTPUT_DIR}/unique_words_by_city.png")
plt.show()

## Summary of Key Findings

### 1. Extreme Geographic Divergence
- **Jaccard similarity: 3-9%** across city pairs
- This is far lower than expected if cities used similar language
- Cities are essentially speaking different "dialects" of real estate marketing

### 2. City-Specific Patterns
- **Chicago:** Emphasizes transit (Metra, CTA) and renovation (rehabbers, tuckpointing)
- **Los Angeles:** Focuses on income potential (ADU) and outdoor living (backyard, hardscaping)
- **New York:** Highlights suburban access (LIRR) and luxury features

### 3. Thematic Coherence
- Word patterns cluster into interpretable categories
- Categories align with known market characteristics:
  - LA climate → outdoor emphasis
  - Chicago transit → walkability emphasis
  - NY density → space/luxury emphasis

### Next Steps
- Run **Notebook 2**: Luxury Language Analysis
- Run **Notebook 3**: City-Specific vs Pooled Models